In [ ]:
import os
os.environ["GROQ_API_KEY"]

In [ ]:
from google.colab import files
upload =files.upload()

Saving English 12 Writing Package.pdf to English 12 Writing Package (2).pdf
Saving e-pwk_unit_2_sample.pdf to e-pwk_unit_2_sample (2).pdf
Saving file-sample_150kB.pdf to file-sample_150kB (2).pdf


In [ ]:
!pip install -q langchain langchain-community langchain-openai faiss-cpu pypdf sentence-transformers

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

pdf_files = [
    "English 12 Writing Package.pdf",
    "e-pwk_unit_2_sample.pdf",
    "file-sample_150kB.pdf"
]

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print("Total Pages:", len(documents))

Total Pages: 62


In [ ]:
print(documents[1].page_content)

Paragraph Writing English 12 
 
Title 
1. What is the title? 
This is the personal title that you choose for your piece of writing that is NOT the name of 
the assignment (which should be instead put in the header). 
2. What does it do? 
It is designed to attract and pique the reader’s interest in your writing. 
3. How do I write one? 
 Many writers think they must title their piece at the start: instead of writing it at the 
beginning, you can 
wait until you have finished your writing and choose a few 
interesting words from your conclusion instead.  
 Once you know your thesis, you can use the ‘main idea’ to help generate a few words 
that encapsulate that main point.  
Example:                                                   The Best Place To Live 
Attention Getter 
1. What 
is the attention getter? 
This is the opening sentence to your paragraph that precedes your thesis or topic sentence 
and is connected to the main idea but generalized. 
2. What does it do? 
It is designed 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))

Total Chunks: 188


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

print("Vector DB Created Successfully")

Vector DB Created Successfully


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
print (retriever)


tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f2ee2124680> search_kwargs={'k': 3}


In [ ]:
docs_list = list(vectorstore.docstore._dict.values())
print(docs_list[0].page_content)

Scott Findley 
School District 43 
Gleneagle Secondary  
Paragraph Writing  
pg. 2 
TAG TIQs CE & Get to 
the Point 
pg. 10 
Get To The Point 
pg. 11 
State/Quote/Clarify 
pg. 12 
Paragraph Response 
Writing 
pg. 16 
Foundational Writing 
pg. 19 
Essay Writing Essential 
Guide 
pg. 21 
Writing Terms 12 
pg. 27 
Academic Writing 
pg. 34 
Writing a Timed Essay 
pg. 35 
Using Quotations 
pg. 36 
Key Verbs in Written 
Response Questions 
pg. 38 
Tone & Formality in 
Academic Writing 
pg. 39 
New Rules for Editing 
pg 41 
Essay Checklists 
pg. 42 
 
 
E12 WRITING 
At the heart of the English curriculum is being able to clearly and 
succinctly express oneself in a written format. Contrary to belief, 
no one is a ‘naturally gifted’ or ‘born writer’; it is a skill that is 
learned through process and practice. Like a muscle, the more you 
properly  exercise your writing abilities, the stronger they will 
become.   
 
So if you need to exercise your writing muscles, where do you 
start?


In [ ]:
for i in range(vectorstore.index.ntotal):
  vector= vectorstore.index.reconstruct(i)
  print(f"vector{i}:")
  print(vector)
  print("-"*50)

Streaming output truncated to the last 5000 lines.
 -1.27226748e-02 -1.44690992e-02 -5.10028526e-02 -5.05157635e-02]
--------------------------------------------------
vector137:
[ 1.50776599e-02  1.15603525e-02  8.52206424e-02  6.96473930e-04
  1.13351075e-02  5.79396933e-02 -5.41684479e-02  4.05142754e-02
 -6.90465942e-02  7.97959268e-02  5.99183142e-03  1.83369052e-02
  2.64596790e-02 -4.95926570e-03  2.20205169e-03  2.70299092e-02
 -6.09016465e-03  4.90726642e-02 -5.70103899e-02 -5.16215935e-02
  6.54829293e-02  2.90228408e-02 -1.53821392e-03  2.98929587e-02
 -5.97220808e-02 -6.52310764e-03  2.77800970e-02 -2.52755731e-02
 -4.70894575e-02 -4.57496708e-03 -7.39802122e-02  1.15778288e-02
  2.04256503e-03  6.18992699e-03 -3.68049815e-02  6.03258573e-02
 -8.46036803e-03  3.38596515e-02  2.99368557e-02 -7.11768866e-02
 -1.06225992e-02 -8.41088817e-02 -5.02461605e-02  6.30254075e-02
  9.40090567e-02 -7.56572932e-02 -1.81077179e-02 -1.80560872e-02
  1.86039601e-02  4.33275104e-02 -6.84631

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt template
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the context below:

{context}

Question: {question}
"""
)

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG pipeline (modern way)
rag_chain = (
    {"context": retriever | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)


query = input("Ask a question: ")

result = rag_chain.invoke(query)

print("\n Answer:\n")
print(result)


Ask a question: lorem ipsum

 Answer:

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Nunc ac faucibus odio.
